# ETS MARL — Sweep Analysis

Counter-part to `scripts/sweep.py`. This notebook is **analysis-only** —
it does not train or instantiate the environment. It loads the CSV logs
produced by a sweep run (typically on Azure) and supports two complementary
modes of investigation:

1. **Top-level comparison across all variants and seeds** — price,
   reward, investment, compliance, secondary market, and newer
   diagnostics over training.
2. **Single-run deep-dive** — pick one `(variant, seed)` and reproduce
   the detailed D / A analysis from `Full Run & Analysis.ipynb` for
   that single run, with the green-investing and collateral-only
   sections removed and collateral folded into the reward
   decomposition panel.


## 1. Setup — Locate Project Root

No training, no environment construction. We only need access to `src/` for a few helpers (paths, optional eval) and to the `configs/` and `results/` folders.


In [ ]:
import os, sys
from pathlib import Path

def _is_project_root(path: str) -> bool:
    return (os.path.isdir(os.path.join(path, "src")) and
            os.path.isdir(os.path.join(path, "configs")))

# Local execution: look for project root relative to notebook location
_local_candidates = [
    "..",             # notebook is in notebooks/ subfolder
    ".",              # notebook opened from project root
    os.path.join("..", ".."),
]
_colab_candidates = [
    os.path.join("Thesis-Energy-Auction", "ets_marl_happo_current"),
    "Thesis-Energy-Auction",
]

_found = _is_project_root(".")
if not _found:
    for _c in _local_candidates + _colab_candidates:
        if _is_project_root(_c):
            os.chdir(_c)
            _found = True
            break
if not _found:
    raise FileNotFoundError(
        f"Cannot find project root (needs src/ + configs/).\n"
        f"CWD: {os.getcwd()}  |  Contents: {os.listdir('.')}"
    )

PROJECT_ROOT = Path.cwd().resolve()
CONFIG_PATH = PROJECT_ROOT / "configs" / "default.yaml"
RESULTS_DIR = PROJECT_ROOT / "results"
sys.path.insert(0, str(PROJECT_ROOT))
print(f"Working directory: {PROJECT_ROOT}")
print(f"Config path: {CONFIG_PATH}")


## 2. Imports


In [ ]:
import os
import re
import yaml
import copy
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)


## 3. Load the Sweep Specification

Point `SWEEP_SPEC` at the YAML you launched the sweep with. The sweep
launcher (`scripts/sweep.py`, see `src/utils/sweep.py` for schema)
deep-merges each variant's `overrides` onto the `base_config` and writes
results under `<output_dir>/<variant_name>/`, with filenames tagged by
`--run-tag <variant>` to keep them unique.


In [ ]:
# Edit this path to point at your sweep spec.
SWEEP_SPEC = PROJECT_ROOT / 'configs' / 'sweeps' / 'example_sweep.yaml'

# Optional override: if your sweep wrote to a different folder than the
# spec's `output_dir`, set this to that absolute path. Otherwise leave None.
OUTPUT_DIR_OVERRIDE = None

from src.utils.sweep import load_sweep_spec, expand_dotted_overrides, deep_merge

spec = load_sweep_spec(str(SWEEP_SPEC))

# Resolve base config (spec-relative path is already resolved by load_sweep_spec).
with open(spec['base_config'], encoding='utf-8') as f:
    base_cfg = yaml.safe_load(f)

if OUTPUT_DIR_OVERRIDE is not None:
    output_dir = Path(OUTPUT_DIR_OVERRIDE)
else:
    output_dir = (PROJECT_ROOT / spec['output_dir']).resolve()

print(f'Sweep spec        : {SWEEP_SPEC}')
print(f'Base config       : {spec["base_config"]}')
print(f'Output dir        : {output_dir}   (exists={output_dir.exists()})')
print(f'Default seeds     : {spec["seeds"]}')
print(f'parallel_workers  : {spec["parallel_workers"]}')
print(f'threads_per_worker: {spec["threads_per_worker"]}')
print(f'#variants         : {len(spec["variants"])}')


### 3a. Variant matrix

Flatten each variant's overrides into a single row so the differences are immediately visible.


In [ ]:
def _flatten_overrides(d, prefix=''):
    """Flatten a nested dict into dotted-path keys."""
    out = {}
    for k, v in d.items():
        key = f'{prefix}.{k}' if prefix else k
        if isinstance(v, dict):
            out.update(_flatten_overrides(v, key))
        else:
            out[key] = v
    return out

rows = []
for v in spec['variants']:
    seeds = v['seeds'] if v['seeds'] is not None else spec['seeds']
    flat = _flatten_overrides(expand_dotted_overrides(v.get('overrides') or {}))
    rows.append({'variant': v['name'], 'n_seeds': len(seeds), 'seeds': seeds, **flat})

variants_df = pd.DataFrame(rows).fillna('—')
variants_df


## 4. Discover Runs

For every `(variant, seed)` pair we look for the per-episode CSV and the
per-year CSV. The sweep launcher writes them with a `--run-tag <variant>`
infix; older single-config runs and locally-run smoke tests do not. We
try the tagged filename first and fall back to the untagged one.


In [ ]:
def _resolve_run_files(variant_name, seed, results_dir):
    """Return (training_log_path, year_log_path) or (None, None) if missing."""
    candidates = [
        # Sweep launcher (tagged)
        (results_dir / f'training_log_{variant_name}_s{seed}.csv',
         results_dir / f'year_log_{variant_name}_s{seed}.csv'),
        # Plain (untagged)
        (results_dir / f'training_log_s{seed}.csv',
         results_dir / f'year_log_s{seed}.csv'),
    ]
    for tlog, ylog in candidates:
        if tlog.exists() and ylog.exists():
            return tlog, ylog
    return None, None

discovery = []
for v in spec['variants']:
    seeds = v['seeds'] if v['seeds'] is not None else spec['seeds']
    variant_dir = output_dir / v['name']
    for seed in seeds:
        tlog, ylog = _resolve_run_files(v['name'], seed, variant_dir)
        discovery.append({
            'variant': v['name'],
            'seed': seed,
            'variant_dir': str(variant_dir),
            'training_log': str(tlog) if tlog else None,
            'year_log': str(ylog) if ylog else None,
            'found': tlog is not None,
        })

discovery_df = pd.DataFrame(discovery)
n_found = int(discovery_df['found'].sum())
print(f'Found logs for {n_found} / {len(discovery_df)} (variant, seed) pairs.')
discovery_df


### 4a. Load all available runs into memory

We also resolve each variant's effective config (base ⊕ overrides) so deep-dive cells can read e.g. `companies.n_agents`.


In [ ]:
def _resolve_variant_config(variant):
    overrides = expand_dotted_overrides(variant.get('overrides') or {})
    return deep_merge(base_cfg, overrides)

runs = {}
variant_configs = {}
for v in spec['variants']:
    variant_configs[v['name']] = _resolve_variant_config(v)

for row in discovery:
    if not row['found']:
        continue
    key = (row['variant'], row['seed'])
    runs[key] = {
        'ep_df': pd.read_csv(row['training_log']),
        'yr_df': pd.read_csv(row['year_log']),
        'config': variant_configs[row['variant']],
        'training_log': row['training_log'],
        'year_log': row['year_log'],
    }

if not runs:
    raise RuntimeError(
        f'No runs loaded. Check that {output_dir} contains the sweep CSVs, '
        'or set OUTPUT_DIR_OVERRIDE in the cell above.'
    )

print(f'Loaded {len(runs)} runs covering {len(set(k[0] for k in runs))} variants.')
for k, v in runs.items():
    print(f'  {k[0]}/s{k[1]:>3} — {len(v["ep_df"]):>5} eps, {len(v["yr_df"]):>6} year-rows')


## 5. Top-Level Comparison Across Runs

All comparison plots use the same pattern:

1. Compute a per-episode scalar metric for every loaded run.
2. Smooth with a rolling mean (`COMPARE_ROLL`) — eyeballed at ~1% of
   `n_episodes` so different sweep sizes look comparable.
3. Plot one thin line per seed (faded) and one bold line per variant
   (mean across seeds), so within-variant noise vs. between-variant
   differences is visible at a glance.


In [ ]:
COMPARE_ROLL = 50          # rolling-mean window for comparison plots
ANALYSIS_TAIL_FRAC = 0.10   # fraction of episodes used for converged-window summary stats

# Stable colour per variant.
_variant_names = [v['name'] for v in spec['variants']]
_palette = sns.color_palette('tab10', n_colors=max(len(_variant_names), 3))
VARIANT_COLOR = {n: _palette[i % len(_palette)] for i, n in enumerate(_variant_names)}

def _agent_cols(df, prefix):
    rx = re.compile(rf'^{re.escape(prefix)}_A(\d+)$')
    return sorted([c for c in df.columns if rx.match(c)],
                  key=lambda c: int(rx.match(c).group(1)))

def _system_metric(ep_df, agg, prefix=None, col=None):
    """Per-episode system-level metric. agg in {'sum', 'mean'}."""
    if col is not None:
        return ep_df[col] if col in ep_df.columns else pd.Series(dtype=float)
    cols = _agent_cols(ep_df, prefix)
    if not cols:
        return pd.Series(dtype=float)
    return ep_df[cols].sum(axis=1) if agg == 'sum' else ep_df[cols].mean(axis=1)

def comparison_frame(metric_fn):
    """Apply metric_fn(ep_df) to each run; return a wide DataFrame indexed by episode."""
    series = {}
    for (variant, seed), run in runs.items():
        s = metric_fn(run['ep_df'])
        if s is None or s.empty:
            continue
        s = s.copy()
        s.index = run['ep_df']['episode'] if 'episode' in run['ep_df'].columns else s.index
        series[(variant, seed)] = s
    if not series:
        return pd.DataFrame()
    return pd.concat(series, axis=1).sort_index()

def plot_comparison(metric_fn, title, ylabel, ax=None, log_y=False, ylim=None):
    df = comparison_frame(metric_fn)
    if df.empty:
        if ax is None:
            fig, ax = plt.subplots(figsize=(8, 4))
        ax.text(0.5, 0.5, f'{title}: no data', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(title)
        return ax

    smooth = df.rolling(COMPARE_ROLL, min_periods=1).mean()

    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 5))

    # Faint per-seed lines.
    for (variant, seed), s in smooth.items():
        ax.plot(s.index, s.values, color=VARIANT_COLOR[variant], alpha=0.25, lw=0.7)

    # Bold per-variant means.
    for variant in [v['name'] for v in spec['variants']]:
        cols = [c for c in smooth.columns if c[0] == variant]
        if not cols:
            continue
        mean_s = smooth[cols].mean(axis=1)
        ax.plot(mean_s.index, mean_s.values, color=VARIANT_COLOR[variant], lw=2.0, label=f'{variant} (n={len(cols)})')

    ax.set_xlabel('Episode')
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    if log_y:
        ax.set_yscale('symlog')
    if ylim is not None:
        ax.set_ylim(*ylim)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8, ncol=min(3, len(spec['variants'])))
    return ax


### 5.1 Carbon Price Over Training

Last-year clearing price per episode. The single number that summarises whether scarcity is biting.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 5))
plot_comparison(lambda df: _system_metric(df, 'mean', col='clearing_price_last'),
                'Clearing Price (last year of episode)', 'EUR/t', ax=axes[0])
plot_comparison(lambda df: _system_metric(df, 'mean', col='price_std'),
                'Intra-Episode Price Std', 'EUR/t', ax=axes[1])
plt.suptitle('C1 — Carbon Price', fontsize=14, y=1.02)
plt.tight_layout(); plt.show()


### 5.2 Reward Over Training

System reward (sum across learning agents) per episode. If `reward_base_A*` columns exist (v6.4+), we plot the constant-trajectory base reward separately from the decaying shaping bonus.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 5))
plot_comparison(lambda df: _system_metric(df, 'sum', prefix='reward'),
                'System Total Reward (Σ reward_A*)', 'reward', ax=axes[0])

# Base reward only (constant trajectory) when available.
def _base_reward(df):
    cols = _agent_cols(df, 'reward_base')
    return df[cols].sum(axis=1) if cols else pd.Series(dtype=float)
plot_comparison(_base_reward, 'System Base Reward (constant trajectory, v6.4+)', 'reward', ax=axes[1])
plt.suptitle('C2 — Reward', fontsize=14, y=1.02)
plt.tight_layout(); plt.show()


### 5.3 Investment Over Training

Total invested capital per episode and the system-mean green fraction.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 5))

def _total_invest(df):
    # Try episode-level first; fall back to year-level via the matching yr_df.
    cols = _agent_cols(df, 'invest_cost')
    if cols:
        return df[cols].sum(axis=1)
    return pd.Series(dtype=float)

plot_comparison(_total_invest,
                'System Investment per Episode (Σ invest_cost_A*)', 'M€', ax=axes[0])

plot_comparison(lambda df: _system_metric(df, 'mean', prefix='green_frac') * 100,
                'Mean Green Fraction Across Agents', 'green frac (%)', ax=axes[1])
plt.suptitle('C3 — Investment & Green Transition', fontsize=14, y=1.02)
plt.tight_layout(); plt.show()


### 5.4 Compliance Over Training

Compliance rate = fraction of `(agent, episode)` cells with `shortfall_A* ≈ 0`. Total system penalty per episode is the second panel.


In [ ]:
def _compliance_rate(df):
    cols = _agent_cols(df, 'shortfall')
    if not cols:
        return pd.Series(dtype=float)
    compliant = (df[cols] <= 1e-6).mean(axis=1)
    return compliant

fig, axes = plt.subplots(1, 2, figsize=(18, 5))
plot_comparison(_compliance_rate, 'Compliance Rate (1 − shortfall>0 fraction)',
                'compliance', ax=axes[0], ylim=(0, 1.02))
plot_comparison(lambda df: _system_metric(df, 'sum', prefix='penalty'),
                'Total Penalty per Episode (M€)', 'penalty', ax=axes[1], log_y=True)
plt.suptitle('C4 — Compliance & Penalty', fontsize=14, y=1.02)
plt.tight_layout(); plt.show()


### 5.5 Secondary Market Over Training

Match rate, traded volume, and the secondary-vs-auction price ratio. The ratio is informative for spotting price-spike regimes (sec ≫ auction) where the v8.5.2/v8.5.3 `expected_remediation_rate_real` proxy kicks in.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
plot_comparison(lambda df: _system_metric(df, 'mean', col='secondary_match_rate'),
                'Secondary Match Rate', 'match rate', ax=axes[0], ylim=(0, 1.02))
plot_comparison(lambda df: _system_metric(df, 'mean', col='secondary_volume'),
                'Secondary Traded Volume', 'Mt', ax=axes[1])

def _price_ratio(df):
    if 'secondary_price' not in df.columns or 'clearing_price_last' not in df.columns:
        return pd.Series(dtype=float)
    return df['secondary_price'] / df['clearing_price_last'].replace(0, np.nan)
plot_comparison(_price_ratio, 'Secondary / Auction Price Ratio', 'ratio', ax=axes[2])
plt.suptitle('C5 — Secondary Market', fontsize=14, y=1.02)
plt.tight_layout(); plt.show()


### 5.6 Newer Diagnostics (added since the original notebook)

These columns / fields are written by recent `train.py` versions; older
logs simply lack them and the relevant panels degrade gracefully.

- **U / D / B / C compliance attribution** (v8.5.1) — partitions every
  `(agent, year)` into mutually exclusive buckets so you can tell whether
  non-compliance was driven by under-bidding (`U`) or by a debt cascade
  from prior carry-forward (`D`), and whether compliance came from the
  bank (`B`) or from the secondary market (`C`).
- **Year-1 advantage clamp diagnostic** (v8.4.1) — `adv_yr1_mean_A1`
  exposes the pre-normalisation year-1 GAE advantage, which the trainer
  now floors at −1.0 to stop early-training year-1 transitions from
  blowing out policy gradients.
- **Split-head investment critic losses** (v8.5) — when
  `ppo.split_invest_head=true`, separate `actor_loss_invest_A*` and
  `critic_loss_invest_A*` columns appear in the episode CSV.


In [ ]:
# UDBC: each column is the per-episode count of (agent, year) cells in that bucket.
# scripts/train.py exposes them as `udbc_U_total_A*` etc.; older logs do not have them.
def _udbc_total(df, bucket):
    cols = _agent_cols(df, f'udbc_{bucket}_total')
    if not cols:
        # Fall back to summing `udbc_<bucket>_A*` if the trainer used a different
        # naming convention in this run.
        cols = _agent_cols(df, f'udbc_{bucket}')
    if not cols:
        return pd.Series(dtype=float)
    return df[cols].sum(axis=1)

fig, axes = plt.subplots(2, 2, figsize=(18, 10))
for ax, bucket, label in [
    (axes[0, 0], 'U', 'Under-bought, non-compliant (U)'),
    (axes[0, 1], 'D', 'Debt-cascade, non-compliant (D)'),
    (axes[1, 0], 'B', 'Bank-covered, compliant (B)'),
    (axes[1, 1], 'C', 'Sec-covered, compliant (C)'),
]:
    plot_comparison(lambda df, b=bucket: _udbc_total(df, b),
                    f'U/D/B/C — {label}', '(agent·year) count', ax=ax)
plt.suptitle('C6a — Compliance attribution buckets (v8.5.1)', fontsize=14, y=1.01)
plt.tight_layout(); plt.show()


In [ ]:
# Split-head invest losses (v8.5) and year-1 advantage clamp diag (v8.4.1).
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

def _split_actor_invest(df):
    cols = _agent_cols(df, 'actor_loss_invest')
    if not cols:
        return pd.Series(dtype=float)
    return df[cols].replace(0, np.nan).mean(axis=1)

def _split_critic_invest(df):
    cols = _agent_cols(df, 'critic_loss_invest')
    if not cols:
        return pd.Series(dtype=float)
    return df[cols].replace(0, np.nan).mean(axis=1)

plot_comparison(_split_actor_invest,
                'Split-head Actor Loss (invest sub-head)', 'actor loss', ax=axes[0])
plot_comparison(_split_critic_invest,
                'Split-head Critic Loss (invest critic)', 'critic loss', ax=axes[1])
plot_comparison(lambda df: df['adv_yr1_mean_A1'] if 'adv_yr1_mean_A1' in df.columns else pd.Series(dtype=float),
                'Year-1 Mean Advantage (A1, pre-normalisation, floor=−1)',
                'advantage', ax=axes[2])
plt.suptitle('C6b — Split-head & year-1 diagnostics', fontsize=14, y=1.02)
plt.tight_layout(); plt.show()


### 5.7 Per-Variant Summary Table (Converged Window)

All metrics averaged over the last `ANALYSIS_TAIL_FRAC` of episodes — a single row per variant for thesis tables.


In [ ]:
def _tail(df):
    if df.empty:
        return df
    n = max(1, int(len(df) * ANALYSIS_TAIL_FRAC))
    return df.tail(n)

summary_rows = []
for (variant, seed), run in runs.items():
    ep, yr = run['ep_df'], run['yr_df']
    ep_tail = _tail(ep)
    yr_tail = yr[yr['episode'].isin(ep_tail['episode'].unique())] if 'episode' in yr.columns else _tail(yr)

    rew_cols = _agent_cols(ep_tail, 'reward')
    sf_cols = _agent_cols(ep_tail, 'shortfall')
    inv_cols = _agent_cols(ep_tail, 'invest_cost')
    gf_cols = _agent_cols(ep_tail, 'green_frac')
    pen_cols = _agent_cols(ep_tail, 'penalty')
    coll_cols = _agent_cols(yr_tail, 'collateral_cost')

    summary_rows.append({
        'variant': variant,
        'seed': seed,
        'price_mean': float(ep_tail['clearing_price_last'].mean()) if 'clearing_price_last' in ep_tail.columns else np.nan,
        'price_std':  float(ep_tail['clearing_price_last'].std())  if 'clearing_price_last' in ep_tail.columns else np.nan,
        'sec_match_rate': float(ep_tail['secondary_match_rate'].mean()) if 'secondary_match_rate' in ep_tail.columns else np.nan,
        'system_reward': float(ep_tail[rew_cols].sum(axis=1).mean()) if rew_cols else np.nan,
        'compliance_rate': float((ep_tail[sf_cols] <= 1e-6).mean(axis=1).mean()) if sf_cols else np.nan,
        'system_penalty': float(ep_tail[pen_cols].sum(axis=1).mean()) if pen_cols else np.nan,
        'system_invest': float(ep_tail[inv_cols].sum(axis=1).mean()) if inv_cols else np.nan,
        'green_frac_mean': float(ep_tail[gf_cols].mean(axis=1).mean()) if gf_cols else np.nan,
        'collateral_year_mean': float(yr_tail[coll_cols].sum(axis=1).mean()) if coll_cols else np.nan,
        'n_episodes': int(len(ep)),
    })

summary_df = pd.DataFrame(summary_rows)

# Aggregate by variant (mean across seeds, with std for the headline metrics).
agg_funcs = {
    'price_mean': ['mean', 'std'],
    'system_reward': ['mean', 'std'],
    'compliance_rate': ['mean', 'std'],
    'system_invest': ['mean', 'std'],
    'green_frac_mean': ['mean', 'std'],
    'collateral_year_mean': ['mean'],
    'sec_match_rate': ['mean'],
    'system_penalty': ['mean'],
    'seed': 'count',
}
by_variant = summary_df.groupby('variant').agg(agg_funcs)
by_variant.columns = ['_'.join([c for c in col if c]) for col in by_variant.columns]
by_variant = by_variant.rename(columns={'seed_count': 'n_seeds'}).round(3)
print('Per-(variant, seed) tail-window summary:')
display(summary_df.round(3))
print('\nAggregated by variant (mean ± std across seeds):')
by_variant


---
## 6. Deep Dive — Pick One Run

Set `PICK_VARIANT` and `PICK_SEED` to the run you want to investigate in
detail. Re-run the cells from here to refresh the deep-dive plots
without recomputing the comparison sections above.


In [ ]:
# Choose which (variant, seed) to deep-dive on. Defaults to the first loaded run.
PICK_VARIANT = next(iter(runs))[0]
PICK_SEED    = next(iter(runs))[1]

# Override here if you want to deep-dive on a different one — e.g.:
# PICK_VARIANT = 'msr_off'
# PICK_SEED    = 2

_run = runs[(PICK_VARIANT, PICK_SEED)]
ep_df = _run['ep_df'].copy()
yr_df = _run['yr_df'].copy()
config = _run['config']
n_agents = int(config.get('companies', {}).get('n_agents', 8))

# Window of converged episodes.
LAST_N = min(200, max(1, len(ep_df) // 4))
last_episodes = ep_df['episode'].unique()[-LAST_N:]
recent_ep = ep_df[ep_df['episode'].isin(last_episodes)].copy()
recent_yr = yr_df[yr_df['episode'].isin(last_episodes)].copy()
ROLL = 50
n_years_per_ep = yr_df.groupby('episode')['year'].count().mode().iloc[0]

agent_colors = [plt.cm.tab10(i) for i in range(n_agents)]
_base_archetypes = ['Coal-Heavy', 'Gas-Dominant', 'Mixed', 'Near-Green']
agent_labels = []
for arch in _base_archetypes:
    for _ in range(n_agents // 4):
        agent_labels.append(f'A{len(agent_labels)+1} ({arch})')
while len(agent_labels) < n_agents:
    agent_labels.append(f'A{len(agent_labels)+1}')

print(f'Deep-diving on: variant={PICK_VARIANT}, seed={PICK_SEED}')
print(f'  {len(ep_df)} episodes, {len(yr_df)} year-rows, {n_years_per_ep} years/episode, {n_agents} learning agents')
print(f'  Converged window: last {LAST_N} episodes')


---
# Part I — Diagnostic Deep Dive (D)

> *Is the simulation behaving correctly?*
>
> Reordered vs. the original notebook: **D5 (stochastic shocks) is last**, and the dedicated green-investing (D7) and collateral (D9 / D9b) sections are removed. Collateral is folded into the reward-decomposition panel in §A5.


### D1. Training Convergence

Check that neural networks are learning stably:
- Losses decreasing
- Entropy decay triggered at the right time
- Rewards stabilizing

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Support both new (v6.4+) and legacy reward logs
reward_cols_total = [f'reward_A{i+1}' for i in range(n_agents) if f'reward_A{i+1}' in ep_df.columns]
reward_cols_base = [f'reward_base_A{i+1}' for i in range(n_agents) if f'reward_base_A{i+1}' in ep_df.columns]
reward_cols_shaping = [f'reward_shaping_A{i+1}' for i in range(n_agents) if f'reward_shaping_A{i+1}' in ep_df.columns]

# 1 — System reward trajectory
ax = axes[0, 0]
if reward_cols_total:
    total_rew = ep_df[reward_cols_total].sum(axis=1)
    total_roll = total_rew.rolling(ROLL, min_periods=1).mean()

    if reward_cols_base:
        total_base = ep_df[reward_cols_base].sum(axis=1)
        base_roll = total_base.rolling(ROLL, min_periods=1).mean()
        ax.plot(ep_df['episode'], base_roll, color='darkgreen', lw=1.5, label='Base reward (constant)')
        ax.plot(ep_df['episode'], total_roll, color='gray', lw=1.1, ls='--', label='Total reward (learning)')

        if reward_cols_shaping:
            total_shaping = ep_df[reward_cols_shaping].sum(axis=1)
            shape_roll = total_shaping.rolling(ROLL, min_periods=1).mean()
            ax.plot(ep_df['episode'], shape_roll, color='darkorange', lw=1.0, label='Shaping bonus (decays)')
        ax.legend(fontsize=7)
        ax.set_title('System Reward Convergence (Base vs Shaping)')
    else:
        ax.plot(ep_df['episode'], total_roll, color='darkgreen', lw=1.2)
        ax.set_title('System Reward Convergence (Total only)')
else:
    ax.text(0.5, 0.5, 'Reward columns not available', ha='center', va='center', transform=ax.transAxes)
ax.set_xlabel('Episode'); ax.set_ylabel('Reward'); ax.grid(True, alpha=0.3)

# 2 — Per-agent reward (constant trajectory when available)
ax = axes[0, 1]
per_agent_cols = reward_cols_base if reward_cols_base else reward_cols_total
for i in range(n_agents):
    col = f'reward_base_A{i+1}' if reward_cols_base else f'reward_A{i+1}'
    if col in ep_df.columns:
        ax.plot(ep_df['episode'], ep_df[col].rolling(ROLL, min_periods=1).mean(),
                color=agent_colors[i], lw=0.8, label=f'A{i+1}')
ax.set_xlabel('Episode'); ax.set_ylabel('Reward')
if reward_cols_base:
    ax.set_title('Per-Agent Base Reward (constant)')
else:
    ax.set_title('Per-Agent Reward (total)')
ax.legend(fontsize=6, ncol=4); ax.grid(True, alpha=0.3)

# 3 — Actor loss
ax = axes[0, 2]
for i in range(n_agents):
    col = f'actor_loss_A{i+1}'
    if col in ep_df.columns:
        vals = ep_df[col].replace(0, np.nan).rolling(ROLL, min_periods=1).mean()
        ax.plot(ep_df['episode'], vals, color=agent_colors[i], lw=0.7)
ax.set_xlabel('Episode'); ax.set_ylabel('Actor Loss')
ax.set_title('Actor Loss'); ax.grid(True, alpha=0.3)

# 4 — Critic loss
ax = axes[1, 0]
for i in range(n_agents):
    col = f'critic_loss_A{i+1}'
    if col in ep_df.columns:
        vals = ep_df[col].replace(0, np.nan).rolling(ROLL, min_periods=1).mean()
        ax.plot(ep_df['episode'], vals, color=agent_colors[i], lw=0.7)
ax.set_xlabel('Episode'); ax.set_ylabel('Critic Loss')
ax.set_title('Critic Loss'); ax.grid(True, alpha=0.3)

# 5 — Entropy coefficient
ax = axes[1, 1]
ax.plot(ep_df['episode'], ep_df['entropy_coef'], color='purple', lw=1)
ax.set_xlabel('Episode'); ax.set_ylabel('Entropy Coef')
ax.set_title('Entropy Coefficient Decay'); ax.grid(True, alpha=0.3)

# 6 — Epsilon
ax = axes[1, 2]
ax.plot(ep_df['episode'], ep_df['epsilon'], color='teal', lw=1)
ax.set_xlabel('Episode'); ax.set_ylabel('ε')
ax.set_title('Exploration ε Decay'); ax.grid(True, alpha=0.3)

plt.suptitle('D1 — Training Convergence Diagnostics', fontsize=14, y=1.01)
plt.tight_layout(); plt.show()

### D2. Carbon Price & Cap Schedule

The clearing price should broadly reflect real EU ETS dynamics:
- Phase 4 prices in the €50–100 range
- Prices rising as the cap tightens under the LRF
- TNAC declining over time as MSR absorbs surplus

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1 — Clearing price over training
ax = axes[0, 0]
ax.plot(ep_df['episode'], ep_df['clearing_price_last'].rolling(ROLL, min_periods=1).mean(),
    color='darkblue', lw=1.2)
ax.axhspan(150, 230, alpha=0.08, color='green', label='Expected EU ETS Range in 2038')
ax.set_xlabel('Episode'); ax.set_ylabel('Price (€/t)')
ax.set_title('Clearing Price Convergence'); ax.legend(); ax.grid(True, alpha=0.3)

# 2 — Price distribution (converged)
ax = axes[0, 1]
prices_conv = recent_yr.groupby('episode')['clearing_price'].last()
ax.hist(prices_conv, bins=40, color='steelblue', edgecolor='white', alpha=0.8)
ax.axvline(prices_conv.median(), color='red', ls='--', label=f'Median={prices_conv.median():.0f}€')
ax.set_xlabel('Clearing Price (€/t)'); ax.set_ylabel('Count')
ax.set_title('Price Distribution (converged)'); ax.legend(); ax.grid(True, alpha=0.3)

# 3 — Auctioned volume vs total emissions (ALL companies incl. bots)
ax = axes[0, 2]
auction_by_yr = recent_yr.groupby('year')['auction_volume'].mean()

emis_cols = sorted(
    [c for c in recent_yr.columns if c.startswith('emissions_A')],
    key=lambda c: int(c.split('_A')[1])
)
total_emis_by_row = recent_yr[emis_cols].sum(axis=1)
emis_by_yr = total_emis_by_row.groupby(recent_yr['year']).mean()

ax.plot(auction_by_yr.index, auction_by_yr.values, 'r-s', ms=5, label='Auction Volume (Mt)', lw=2)
ax.plot(emis_by_yr.index, emis_by_yr.values, 'b-o', ms=4, label='Total Emissions (Mt)', lw=1.5)
ax.fill_between(auction_by_yr.index, auction_by_yr.values, emis_by_yr.values, alpha=0.15, color='gray')
ax.set_xlabel('Year'); ax.set_ylabel('Mt CO2')
ax.set_title(f'Auction Volume vs Emissions (converged mean, {len(emis_cols)} firms)')
ax.legend(); ax.grid(True, alpha=0.3)

# 4 — Within-episode price trajectory
ax = axes[1, 0]
sample_eps = sorted(recent_yr['episode'].unique()[-20:])
for ep in sample_eps:
    ep_data = recent_yr[recent_yr['episode'] == ep]
    ax.plot(ep_data['year'], ep_data['clearing_price'], color='steelblue', alpha=0.15, lw=0.8)
mean_price_yr = recent_yr.groupby('year')['clearing_price'].mean()
ax.plot(mean_price_yr.index, mean_price_yr.values, 'darkblue', lw=2.5, label='Mean')
ax.set_xlabel('Year'); ax.set_ylabel('Price (€/t)')
ax.set_title('Within-Episode Price Trajectory'); ax.legend(); ax.grid(True, alpha=0.3)

# 5 — TNAC over years
ax = axes[1, 1]
tnac_yr = recent_yr.groupby('year')['tnac'].agg(['mean', 'std'])
std_tnac = tnac_yr['std'].fillna(0.0)
ax.plot(tnac_yr.index, tnac_yr['mean'], 'darkorange', lw=2, marker='o', ms=4)
ax.fill_between(tnac_yr.index, tnac_yr['mean'] - std_tnac, tnac_yr['mean'] + std_tnac,
        alpha=0.2, color='orange')
ax.set_xlabel('Year'); ax.set_ylabel('TNAC (Mt)')
ax.set_title('Allowances in Circulation (mean±1σ)'); ax.grid(True, alpha=0.3)

# 6 — MSR reserve
ax = axes[1, 2]
msr_yr = recent_yr.groupby('year')['msr_reserve'].mean()
ax.bar(msr_yr.index, msr_yr.values, color='mediumpurple', alpha=0.8)
ax.set_xlabel('Year'); ax.set_ylabel('MSR Reserve (Mt)')
ax.set_title('Market Stability Reserve (converged mean)'); ax.grid(True, alpha=0.3)

plt.suptitle('D2 — Carbon Price & Auction Supply Diagnostics', fontsize=14, y=1.01)
plt.tight_layout(); plt.show()

### D3. Auction Mechanics

Sanity checks for the uniform-price auction:
- Allocations should sum to roughly the auctioned volume
- Bid prices should bracket the clearing price
- No persistent auction failures

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1 — Total allocation vs auction volume
ax = axes[0, 0]
alloc_cols = [f'alloc_A{i+1}' for i in range(n_agents)]
alloc_total = recent_yr[alloc_cols].sum(axis=1)
ax.scatter(recent_yr['auction_volume'], alloc_total, alpha=0.05, s=8, color='steelblue')
lims = [0, max(recent_yr['auction_volume'].max(), alloc_total.max()) * 1.1]
ax.plot(lims, lims, 'r--', lw=1, label='Perfect match')
ax.set_xlabel('Auction Volume (Mt)'); ax.set_ylabel('Total Allocated (Mt)')
ax.set_title('Allocation vs Supply'); ax.legend(); ax.grid(True, alpha=0.3)

# 2 — Per-agent allocation distribution
ax = axes[0, 1]
alloc_data = [recent_yr[f'alloc_A{i+1}'].values for i in range(n_agents)]
bp = ax.boxplot(alloc_data, tick_labels=[f'A{i+1}' for i in range(n_agents)],
                patch_artist=True, showfliers=False)
for i, patch in enumerate(bp['boxes']):
    patch.set_facecolor(agent_colors[i])
ax.set_ylabel('Allocation (Mt)'); ax.set_title('Allocation Distribution (converged)')
ax.grid(True, alpha=0.3)

# 3 — Bid price vs clearing price
ax = axes[0, 2]
for i in range(n_agents):
    bid_roll = ep_df[f'bid_price_A{i+1}'].rolling(ROLL, min_periods=1).mean()
    ax.plot(ep_df['episode'], bid_roll, color=agent_colors[i], lw=0.8, label=f'A{i+1}')
clr_roll = ep_df['clearing_price_last'].rolling(ROLL, min_periods=1).mean()
ax.plot(ep_df['episode'], clr_roll, 'k--', lw=2, label='Clearing')
ax.set_xlabel('Episode'); ax.set_ylabel('Price (\u20ac/t)')
ax.set_title('Bid Prices vs Clearing Price'); ax.legend(fontsize=6, ncol=3); ax.grid(True, alpha=0.3)

# 4 — Auction cost per agent (converged)
ax = axes[1, 0]
acost_means = [recent_yr.groupby('episode')[f'auction_cost_A{i+1}'].sum().mean() for i in range(n_agents)]
bars = ax.bar([f'A{i+1}' for i in range(n_agents)], acost_means,
              color=agent_colors)
ax.set_ylabel('Mean Auction Cost / Episode (M\u20ac)')
ax.set_title('Auction Cost by Agent (converged)'); ax.grid(True, alpha=0.3)

# 5 — Under-allocation frequency
ax = axes[1, 1]
# Check how often total allocation < 90% of auction volume
under = alloc_total < 0.9 * recent_yr['auction_volume']
under_rate_yr = under.groupby(recent_yr['year']).mean()
ax.bar(under_rate_yr.index, under_rate_yr.values * 100, color='salmon', alpha=0.8)
ax.set_xlabel('Year'); ax.set_ylabel('Under-allocation Rate (%)')
ax.set_title('Auction Under-Subscription Frequency'); ax.grid(True, alpha=0.3)

# 6 — Price volatility over training
ax = axes[1, 2]
ax.plot(ep_df['episode'], ep_df['price_std'].rolling(ROLL, min_periods=1).mean(),
        color='teal', lw=1)
ax.set_xlabel('Episode'); ax.set_ylabel('Within-Episode Price Std (\u20ac)')
ax.set_title('Intra-Episode Price Volatility'); ax.grid(True, alpha=0.3)

plt.suptitle('D3 \u2014 Auction Mechanics Diagnostics', fontsize=14, y=1.01)
plt.tight_layout(); plt.show()

### D4. Secondary Market Mechanics

Does the double-auction secondary market function properly?
- Reasonable match rates (not all-or-nothing)
- Secondary prices tracking auction prices
- Both buyers and sellers participating

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1 — Match rate over training
ax = axes[0, 0]
ax.plot(ep_df['episode'], ep_df['secondary_match_rate'].rolling(ROLL, min_periods=1).mean(),
        color='coral', lw=1.2)
ax.set_xlabel('Episode'); ax.set_ylabel('Match Rate')
ax.set_title('Secondary Market Match Rate'); ax.grid(True, alpha=0.3)

# 2 — Secondary volume over training
ax = axes[0, 1]
ax.plot(ep_df['episode'], ep_df['secondary_volume'].rolling(ROLL, min_periods=1).mean(),
        color='steelblue', lw=1.2)
ax.set_xlabel('Episode'); ax.set_ylabel('Volume (Mt)')
ax.set_title('Secondary Market Volume'); ax.grid(True, alpha=0.3)

# 3 — Secondary vs Auction price
ax = axes[0, 2]
sec_mean_yr = recent_yr.groupby('year')['secondary_price'].mean()
auc_mean_yr = recent_yr.groupby('year')['clearing_price'].mean()
ax.plot(auc_mean_yr.index, auc_mean_yr.values, 'b-o', ms=4, label='Auction', lw=1.5)
ax.plot(sec_mean_yr.index, sec_mean_yr.values, 'r-s', ms=4, label='Secondary', lw=1.5)
ax.set_xlabel('Year'); ax.set_ylabel('Price (\u20ac/t)')
ax.set_title('Auction vs Secondary Price (converged)'); ax.legend(); ax.grid(True, alpha=0.3)

# 4 — Net trade by agent by year (who buys/sells)
ax = axes[1, 0]
for i in range(n_agents):
    trade_by_yr = recent_yr.groupby('year')[f'trade_qty_A{i+1}'].mean()
    ax.plot(trade_by_yr.index, trade_by_yr.values, color=agent_colors[i],
            marker='o', ms=3, label=f'A{i+1}', lw=1)
ax.axhline(0, color='black', ls='-', lw=0.5)
ax.set_xlabel('Year'); ax.set_ylabel('Net Trade Qty (Mt)')
ax.set_title('Mean Trade Position by Year (converged)')
ax.legend(fontsize=6, ncol=4); ax.grid(True, alpha=0.3)

# 5 — Buy/sell balance: count of buyers vs sellers per year
ax = axes[1, 1]
buy_count_yr = []
sell_count_yr = []
for yr in range(n_years_per_ep):
    yr_data = recent_yr[recent_yr['year'] == yr]
    n_buy = sum((yr_data[f'trade_qty_A{i+1}'] > 0.01).mean() for i in range(n_agents))
    n_sell = sum((yr_data[f'trade_qty_A{i+1}'] < -0.01).mean() for i in range(n_agents))
    buy_count_yr.append(n_buy)
    sell_count_yr.append(n_sell)
x = np.arange(n_years_per_ep)
ax.bar(x - 0.15, buy_count_yr, 0.3, color='forestgreen', alpha=0.8, label='Buyers')
ax.bar(x + 0.15, sell_count_yr, 0.3, color='firebrick', alpha=0.8, label='Sellers')
ax.set_xlabel('Year'); ax.set_ylabel('Avg # Agents')
ax.set_title('Buyers vs Sellers per Year'); ax.legend(); ax.grid(True, alpha=0.3)

# 6 — Secondary net cost distribution
ax = axes[1, 2]
snet_data = []
for i in range(n_agents):
    ep_snet = recent_yr.groupby('episode')[f'secondary_net_A{i+1}'].sum()
    snet_data.append(ep_snet.values)
bp = ax.boxplot(snet_data, tick_labels=[f'A{i+1}' for i in range(n_agents)],
                patch_artist=True, showfliers=False)
for i, patch in enumerate(bp['boxes']):
    patch.set_facecolor(agent_colors[i])
ax.axhline(0, color='black', ls='--', lw=0.5)
ax.set_ylabel('Net Secondary Cost / Episode (M\u20ac)')
ax.set_title('Secondary Net Cost (+paid / -received)')
ax.grid(True, alpha=0.3)

plt.suptitle('D4 \u2014 Secondary Market Diagnostics', fontsize=14, y=1.01)
plt.tight_layout(); plt.show()

### D6. Compliance & Penalty Mechanics

Verify that compliance mechanics work:
- Shortfalls are detected and penalized
- Penalties are proportional to shortfall
- Banking (carry-over) functions correctly

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1 — Shortfall rate over training
ax = axes[0, 0]
for i in range(n_agents):
    sf = (ep_df[f'shortfall_A{i+1}'] > 1e-6).astype(float).rolling(ROLL, min_periods=1).mean()
    ax.plot(ep_df['episode'], sf * 100, color=agent_colors[i], lw=0.8, label=f'A{i+1}')
ax.set_xlabel('Episode'); ax.set_ylabel('Episodes with Shortfall (%)')
ax.set_title('Shortfall Frequency'); ax.legend(fontsize=6, ncol=4); ax.grid(True, alpha=0.3)

# 2 - Which year do shortfalls occur? (converged episodes)
ax = axes[0, 1]
shortfall_by_year = {}
for yr in sorted(recent_yr['year'].unique()):
    yr_rows = recent_yr[recent_yr['year'] == yr]
    rates = [(yr_rows[f'shortfall_A{i+1}'] > 1e-6).mean() for i in range(n_agents)]
    shortfall_by_year[yr] = rates
years = sorted(shortfall_by_year.keys())
x = np.arange(len(years))
bar_w = 0.8 / max(n_agents, 1)
for i in range(n_agents):
    vals = [shortfall_by_year[yr][i] * 100 for yr in years]
    ax.bar(x + i * bar_w - 0.4 + bar_w/2, vals, width=bar_w,
           color=agent_colors[i], alpha=0.75, label=f'A{i+1}')
ax.set_xticks(x); ax.set_xticklabels([f'Yr {y}' for y in years], fontsize=7)
ax.set_xlabel('Year within Episode'); ax.set_ylabel('Episodes with Shortfall (%)')
ax.set_title('Which Year Do Shortfalls Occur? (converged)')
ax.legend(fontsize=6, ncol=4); ax.grid(True, alpha=0.3, axis='y')

# 3 — Holdings by year (banking)
ax = axes[0, 2]
for i in range(n_agents):
    hold_yr = recent_yr.groupby('year')[f'holdings_A{i+1}'].mean()
    ax.plot(hold_yr.index, hold_yr.values, color=agent_colors[i], marker='o', ms=3,
            label=f'A{i+1}', lw=1)
ax.set_xlabel('Year'); ax.set_ylabel('Holdings (Mt)')
ax.set_title('Banking: Holdings by Year (converged)')
ax.legend(fontsize=6, ncol=4); ax.grid(True, alpha=0.3)

# 4 — Compliance surplus distribution
ax = axes[1, 0]
if f'compliance_surplus_A1' in recent_yr.columns:
    surp_data = [recent_yr.groupby('episode')[f'compliance_surplus_A{i+1}'].mean().values
                 for i in range(n_agents)]
    bp = ax.boxplot(surp_data, tick_labels=[f'A{i+1}' for i in range(n_agents)],
                    patch_artist=True, showfliers=False)
    for i, patch in enumerate(bp['boxes']):
        patch.set_facecolor(agent_colors[i])
    ax.axhline(0, color='red', ls='--', lw=0.5)
    ax.set_ylabel('Mean Compliance Surplus (Mt)')
    ax.set_title('Compliance Surplus (+over / -under)')
else:
    ax.text(0.5, 0.5, 'compliance_surplus not logged', ha='center', va='center',
            transform=ax.transAxes)
ax.grid(True, alpha=0.3)

# 5 — Total penalty over training
ax = axes[1, 1]
pen_cols = [f'penalty_A{i+1}' for i in range(n_agents)]
total_pen = ep_df[pen_cols].sum(axis=1)
ax.plot(ep_df['episode'], total_pen.rolling(ROLL, min_periods=1).mean(), color='firebrick', lw=1.2)
ax.set_xlabel('Episode'); ax.set_ylabel('Total Penalty (M€)')
ax.set_title('System-Wide Penalty Trend'); ax.grid(True, alpha=0.3)

# 6 — Carry-forward: bank_end vs bank_start
ax = axes[1, 2]
if 'bank_end_A1' in recent_yr.columns and 'bank_start_A1' in recent_yr.columns:
    for i in range(min(8, n_agents)):
        bs = recent_yr.groupby('year')[f'bank_start_A{i+1}'].mean()
        be = recent_yr.groupby('year')[f'bank_end_A{i+1}'].mean()
        ax.plot(bs.index, bs.values, color=agent_colors[i], ls='--', lw=1, alpha=0.7)
        ax.plot(be.index, be.values, color=agent_colors[i], ls='-', lw=1.5,
                label=f'A{i+1} (solid=end, dash=start)')
    ax.set_xlabel('Year'); ax.set_ylabel('Bank (Mt)')
    ax.set_title('Banking: Start vs End of Year (first 8 agents)')
    ax.legend(fontsize=6, ncol=2)
else:
    ax.text(0.5, 0.5, 'bank_end not logged', ha='center', va='center', transform=ax.transAxes)
ax.grid(True, alpha=0.3)

plt.suptitle('D6 — Compliance & Penalty Diagnostics', fontsize=14, y=1.01)
plt.tight_layout(); plt.show()

### D8. Final-Year Invested Assets, Queue State, and Liquidation Payoff

> This cell now prioritizes liquidation values logged directly from the environment reward computation into year_log CSV.

> Fallback behavior:
> - If `terminal_*_value_A*` columns are absent (older logs), bank liquidation is reconstructed from bank_end × clearing_price / 1000.
> - Queue liquidation is shown only if logged columns exist.

In [ ]:
# ── D8: Final-Year Invested Assets, Queue State, and Liquidation Payoff ─────
reward_cols_all = sorted(
    [c for c in recent_yr.columns if c.startswith('reward_A')],
    key=lambda x: int(x.split('_A')[1])
)
n_logged = max(int(c.split('_A')[1]) for c in reward_cols_all) if reward_cols_all else n_agents
agent_ids_logged = list(range(1, n_logged + 1))

y_last = int(recent_yr['year'].max())
final_slice = recent_yr[recent_yr['year'] == y_last].copy()

reward_cfg = config.get('reward', {})
terminal_bank_enabled = bool(reward_cfg.get('terminal_bank_value', False))
terminal_queue_enabled = bool(reward_cfg.get('terminal_queue_value', False))

agent_lbls_d8 = []
final_reward_vals = []
bank_end_vals = []
queue_end_vals = []
bank_liq_vals = []
queue_liq_vals = []
total_liq_vals = []

for i in agent_ids_logged:
    a = f'A{i}'
    r_col = f'reward_A{i}'
    b_col = f'bank_end_A{i}'
    q_col = f'queue_size_A{i}'
    b_liq_col = f'terminal_bank_value_A{i}'
    q_liq_col = f'terminal_queue_value_A{i}'
    t_liq_col = f'terminal_liquidation_value_A{i}'

    if r_col not in final_slice.columns:
        continue

    rf = float(final_slice[r_col].mean())
    be = float(final_slice[b_col].mean()) if b_col in final_slice.columns else np.nan
    qe = float(final_slice[q_col].mean()) if q_col in final_slice.columns else np.nan

    if t_liq_col in final_slice.columns:
        t_liq = float(final_slice[t_liq_col].mean())
        b_liq = float(final_slice[b_liq_col].mean()) if b_liq_col in final_slice.columns else np.nan
        q_liq = float(final_slice[q_liq_col].mean()) if q_liq_col in final_slice.columns else np.nan
    else:
        # Fallback for old logs.
        if terminal_bank_enabled and b_col in final_slice.columns and 'clearing_price' in final_slice.columns:
            b_liq = float((final_slice[b_col] * final_slice['clearing_price'] / 1000.0).mean())
        else:
            b_liq = 0.0
        q_liq = 0.0
        t_liq = b_liq + q_liq

    agent_lbls_d8.append(a)
    final_reward_vals.append(rf)
    bank_end_vals.append(be)
    queue_end_vals.append(qe)
    bank_liq_vals.append(b_liq)
    queue_liq_vals.append(q_liq)
    total_liq_vals.append(t_liq)

fig, axes = plt.subplots(2, 3, figsize=(19, 11))
x = np.arange(len(agent_lbls_d8))

ax = axes[0, 0]
ax.bar(x, bank_end_vals, color='darkorange', alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(agent_lbls_d8, rotation=45)
ax.set_xlabel('Agent'); ax.set_ylabel('Bank End (Mt)')
ax.set_title('Final-Year Mean Banked Allowances'); ax.grid(True, alpha=0.3)

ax = axes[0, 1]
ax.bar(x, queue_end_vals, color='teal', alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(agent_lbls_d8, rotation=45)
ax.set_xlabel('Agent'); ax.set_ylabel('Queue Items')
ax.set_title('Final-Year Mean Queue Size'); ax.grid(True, alpha=0.3)

ax = axes[0, 2]
w = 0.27
ax.bar(x - w, bank_liq_vals, w, color='slateblue', alpha=0.85, label='Bank liquidation')
ax.bar(x,     queue_liq_vals, w, color='seagreen', alpha=0.85, label='Queue liquidation')
ax.bar(x + w, total_liq_vals, w, color='purple', alpha=0.75, label='Total liquidation')
ax.set_xticks(x); ax.set_xticklabels(agent_lbls_d8, rotation=45)
ax.set_xlabel('Agent'); ax.set_ylabel('Reward Units')
ax.set_title('Final-Year Liquidation Terms by Agent'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

ax = axes[1, 0]
rew_minus_liq = [r - l for r, l in zip(final_reward_vals, total_liq_vals)]
w = 0.38
ax.bar(x - w / 2, final_reward_vals, w, color='firebrick', alpha=0.8, label='Final reward')
ax.bar(x + w / 2, rew_minus_liq,    w, color='steelblue', alpha=0.8, label='Final reward minus liquidation')
ax.axhline(0, color='black', lw=0.8)
ax.set_xticks(x); ax.set_xticklabels(agent_lbls_d8, rotation=45)
ax.set_xlabel('Agent'); ax.set_ylabel('Reward')
ax.set_title('Final Reward with/without Liquidation Terms'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

ax = axes[1, 1]
if total_liq_vals:
    ax.scatter(total_liq_vals, final_reward_vals, s=36, color='purple', alpha=0.85)
    for xv, yv, lbl in zip(total_liq_vals, final_reward_vals, agent_lbls_d8):
        ax.text(xv, yv, lbl, fontsize=8, alpha=0.75)
    if len(total_liq_vals) > 2 and np.std(total_liq_vals) > 1e-12:
        m = np.polyfit(total_liq_vals, final_reward_vals, 1)
        xfit = np.linspace(min(total_liq_vals), max(total_liq_vals), 50)
        ax.plot(xfit, m[0] * xfit + m[1], 'k--', lw=1)
ax.set_xlabel('Total liquidation term'); ax.set_ylabel('Final-year reward')
ax.set_title('Total Liquidation Term vs Final Reward'); ax.grid(True, alpha=0.3)

ax = axes[1, 2]
qarr = np.array(queue_liq_vals, dtype=float)
if np.isfinite(qarr).any() and np.nansum(np.abs(qarr)) > 0:
    ax.scatter(queue_liq_vals, final_reward_vals, s=36, color='seagreen', alpha=0.85)
    for xv, yv, lbl in zip(queue_liq_vals, final_reward_vals, agent_lbls_d8):
        ax.text(xv, yv, lbl, fontsize=8, alpha=0.75)
    ax.set_title('Queue Liquidation Term vs Final Reward')
else:
    ax.text(0.5, 0.5, 'Queue liquidation absent / all zero', ha='center', va='center', transform=ax.transAxes)
    ax.set_title('Queue Liquidation — no signal')
ax.set_xlabel('Queue liquidation term'); ax.set_ylabel('Final-year reward'); ax.grid(True, alpha=0.3)

plt.suptitle('D8 — Final-Year Assets, Queue & Liquidation', fontsize=14, y=1.01)
plt.tight_layout(); plt.show()


### D5. Stochastic Shocks (P5 & P6)

Verify that emission demand shocks (P5) and construction uncertainty (P6) are
active and producing the intended distributions.

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 14))

shock_cols = [f'emission_shock_A{i+1}' for i in range(n_agents)]
cf_cols = [f'cf_shock_A{i+1}' for i in range(n_agents)]
cancel_cols = [f'cancellation_A{i+1}' for i in range(n_agents)]

has_shocks = any(c in recent_yr.columns for c in shock_cols) and recent_yr[shock_cols[0]].std() > 1e-6

# 1 — P5 shock distribution
ax = axes[0, 0]
if has_shocks:
    all_shocks = pd.concat([recent_yr[c] * 100 for c in shock_cols if c in recent_yr.columns])
    ax.hist(all_shocks, bins=60, color='steelblue', edgecolor='white', alpha=0.8, density=True)
    ax.set_xlabel('Emission Shock ε (%)')
    ax.set_title(f'P5 Shock Distribution (σ={all_shocks.std():.1f}%)')
else:
    ax.text(0.5, 0.5, 'P5 shocks inactive', ha='center', va='center', transform=ax.transAxes)
    ax.set_title('P5 — Inactive')
ax.grid(True, alpha=0.3)

# 2 — P5 cross-agent correlation
ax = axes[0, 1]
if has_shocks and n_agents >= 2:
    s1 = recent_yr[shock_cols[0]] * 100
    s2 = recent_yr[shock_cols[1]] * 100
    ax.scatter(s1, s2, alpha=0.05, s=5, color='steelblue')
    corr = s1.corr(s2)
    ax.set_xlabel('A1 shock (%)'); ax.set_ylabel('A2 shock (%)')
    ax.set_title(f'P5 Cross-Agent Correlation (ρ={corr:.2f})')
else:
    ax.text(0.5, 0.5, 'P5 inactive or <2 agents', ha='center', va='center', transform=ax.transAxes)
ax.grid(True, alpha=0.3)

# 3 — P6 CF noise
ax = axes[1, 0]
has_cf = any(c in recent_yr.columns for c in cf_cols) and recent_yr[cf_cols[0]].std() > 1e-8
if has_cf:
    all_cf = pd.concat([recent_yr[c] * 100 for c in cf_cols if c in recent_yr.columns])
    ax.hist(all_cf, bins=60, color='seagreen', edgecolor='white', alpha=0.8, density=True)
    ax.set_xlabel('CF Noise (%)')
    ax.set_title(f'P6 Capacity-Factor Noise (σ={all_cf.std():.1f}%)')
else:
    ax.text(0.5, 0.5, 'P6 CF-noise inactive', ha='center', va='center', transform=ax.transAxes)
    ax.set_title('P6 CF — Inactive')
ax.grid(True, alpha=0.3)

# 4 — P6 cancellations
ax = axes[1, 1]
has_cancel = any(c in recent_yr.columns for c in cancel_cols) and recent_yr[cancel_cols[0]].sum() > 0
if has_cancel:
    cancel_per_ep = recent_yr.groupby('episode')[cancel_cols].sum().sum(axis=1)
    ax.hist(cancel_per_ep, bins=30, color='coral', edgecolor='white', alpha=0.8)
    ax.set_xlabel('Total Cancellations / Episode')
    ax.set_title(f'P6 Project Cancellations (mean={cancel_per_ep.mean():.1f}/ep)')
else:
    ax.text(0.5, 0.5, 'P6 cancellations inactive', ha='center', va='center', transform=ax.transAxes)
    ax.set_title('P6 Cancellations — Inactive')
ax.grid(True, alpha=0.3)

# 5 — Inflation rate distribution
ax = axes[2, 0]
has_infl = 'inflation_rate' in recent_yr.columns and recent_yr['inflation_rate'].notna().any()
if has_infl:
    infl_pct = recent_yr['inflation_rate'] * 100
    ax.hist(infl_pct, bins=40, color='mediumpurple', edgecolor='white', alpha=0.8, density=True)
    ax.axvline(infl_pct.mean(), color='black', ls='--', lw=1.2, label=f'Mean={infl_pct.mean():.2f}%')
    ax.set_xlabel('Inflation Rate (%)')
    ax.set_title(f'Inflation Distribution (σ={infl_pct.std():.2f}%-pt)')
    ax.legend()
else:
    ax.text(0.5, 0.5, 'Inflation not logged in year_log', ha='center', va='center', transform=ax.transAxes)
    ax.set_title('Inflation — Not Available')
ax.grid(True, alpha=0.3)

# 6 — Inflation profile by year
ax = axes[2, 1]
if has_infl:
    infl_by_year = recent_yr.groupby('year')['inflation_rate'].agg(['mean', 'std']) * 100
    ax.plot(infl_by_year.index, infl_by_year['mean'], color='purple', marker='o', lw=2)
    std_fill = infl_by_year['std'].fillna(0.0)
    ax.fill_between(
        infl_by_year.index,
        infl_by_year['mean'] - std_fill,
        infl_by_year['mean'] + std_fill,
        color='plum', alpha=0.25
    )
    ax.set_xlabel('Year')
    ax.set_ylabel('Inflation Rate (%)')
    ax.set_title('Inflation by Year (mean±1σ)')
else:
    ax.text(0.5, 0.5, 'Inflation not logged in year_log', ha='center', va='center', transform=ax.transAxes)
    ax.set_title('Inflation by Year — Not Available')
ax.grid(True, alpha=0.3)

plt.suptitle('D5 — Stochastic Shock & Inflation Diagnostics (P5/P6 + Inflation)', fontsize=14, y=1.01)
plt.tight_layout(); plt.show()

# Summary
print(f'P5 emission shock active: {has_shocks}')
print(f'P6 CF-noise active:       {has_cf}')
print(f'P6 cancellation active:   {has_cancel}')
print(f'Inflation logged:         {has_infl}')
if has_infl:
    infl_pct = recent_yr['inflation_rate'] * 100
    print(f'Inflation mean:           {infl_pct.mean():.3f}%')
    print(f'Inflation std:            {infl_pct.std():.3f}%-pt')

---
# Part II — Strategy Deep Dive (A)

> *What strategies did the agents learn and why?*

Drops the bot-comparison section (A7 / A7a) and the standalone summary /
troubleshooting digest. The standalone collateral panel (A5b) is removed;
collateral is folded into the cost-breakdown panel of §A5.


### A1. Auction Bidding Strategy

How agents bid in the primary EUA auction:
- **Price discovery** — do bids converge to a common clearing price?
- **Quantity strategy** — do agents bid for what they need or over/under-bid?
- **Archetype differences** — do coal-heavy agents bid differently from near-green?

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))

# 1 — Bid price evolution per agent
ax = axes[0, 0]
for i in range(n_agents):
    ax.plot(ep_df['episode'], ep_df[f'bid_price_A{i+1}'].rolling(ROLL, min_periods=1).mean(),
            color=agent_colors[i], lw=0.9, label=agent_labels[i])
ax.set_xlabel('Episode'); ax.set_ylabel('Mean Bid Price (\u20ac/t)')
ax.set_title('Bid Price Evolution'); ax.legend(fontsize=6, ncol=2); ax.grid(True, alpha=0.3)

# 2 — Bid price distribution (converged)
ax = axes[0, 1]
bid_data = [recent_ep[f'bid_price_A{i+1}'].values for i in range(n_agents)]
parts = ax.violinplot(bid_data, positions=range(n_agents), showmedians=True, showextrema=False)
for i, pc in enumerate(parts['bodies']):
    pc.set_facecolor(agent_colors[i]); pc.set_alpha(0.7)
ax.set_xticks(range(n_agents)); ax.set_xticklabels([f'A{i+1}' for i in range(n_agents)])
ax.set_ylabel('Bid Price (\u20ac/t)')
ax.set_title('Bid Price Distribution (converged)'); ax.grid(True, alpha=0.3)

# 3 — Allocation vs Emissions scatter (converged)
ax = axes[0, 2]
for i in range(n_agents):
    allocs = recent_yr.groupby('episode')[f'alloc_A{i+1}'].mean()
    emiss = recent_yr.groupby('episode')[f'emissions_A{i+1}'].mean()
    ax.scatter(allocs, emiss, alpha=0.15, s=12, color=agent_colors[i], label=f'A{i+1}')
lims = [0, max(ax.get_xlim()[1], ax.get_ylim()[1])]
ax.plot(lims, lims, 'k--', lw=0.8, alpha=0.5, label='Alloc=Emissions')
ax.set_xlabel('Mean Allocation (Mt)'); ax.set_ylabel('Mean Emissions (Mt)')
ax.set_title('Allocation vs Emissions Need')
ax.legend(fontsize=6, ncol=3); ax.grid(True, alpha=0.3)

# 4 — Bid quantity evolution
ax = axes[1, 0]
for i in range(n_agents):
    alloc_roll = recent_yr.groupby('episode')[f'alloc_A{i+1}'].mean()
    alloc_smooth = alloc_roll.rolling(ROLL // 2, min_periods=1).mean()
    ax.plot(alloc_smooth.index, alloc_smooth.values, color=agent_colors[i], lw=0.9, label=f'A{i+1}')
ax.set_xlabel('Episode'); ax.set_ylabel('Mean Allocation (Mt)')
ax.set_title('Allocation Over Training (proxy for bid qty)'); ax.legend(fontsize=6, ncol=4)
ax.grid(True, alpha=0.3)

# 5 — Bid price by year within episode (converged)
ax = axes[1, 1]
for i in range(n_agents):
    bid_yr = recent_yr.groupby('year')[f'bid_price_A{i+1}'].mean()
    ax.plot(bid_yr.index, bid_yr.values, color=agent_colors[i], marker='o', ms=3,
            lw=1, label=f'A{i+1}')
ax.set_xlabel('Year'); ax.set_ylabel('Bid Price (\u20ac/t)')
ax.set_title('Within-Episode Bid Price by Year')
ax.legend(fontsize=6, ncol=4); ax.grid(True, alpha=0.3)

# 6 — Coverage ratio: allocation / emissions
ax = axes[1, 2]
cov_data = []
for i in range(n_agents):
    a = recent_yr[f'alloc_A{i+1}']
    e = recent_yr[f'emissions_A{i+1}'].replace(0, np.nan)
    cov_data.append((a / e).dropna().values)
bp = ax.boxplot(cov_data, tick_labels=[f'A{i+1}' for i in range(n_agents)],
                patch_artist=True, showfliers=False)
for i, patch in enumerate(bp['boxes']):
    patch.set_facecolor(agent_colors[i])
ax.axhline(1.0, color='red', ls='--', lw=0.8, label='Full coverage')
ax.set_ylabel('Allocation / Emissions')
ax.set_title('Coverage Ratio (converged)'); ax.legend(); ax.grid(True, alpha=0.3)

plt.suptitle('A1 \u2014 Auction Bidding Strategy', fontsize=14, y=1.01)
plt.tight_layout(); plt.show()

### A2. Secondary Market Strategy

How agents trade on the secondary market:
- **Role evolution** — who becomes a net buyer vs seller over training?
- **Year-by-year patterns** — do roles shift within an episode as the cap tightens?
- **Price premiums** — secondary vs auction price spread
- **Cost/revenue** — who profits from secondary trading?

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(16, 16))

# 1 — Net trade position over training (rolling)
ax = axes[0, 0]
for i in range(n_agents):
    net_trade = yr_df.groupby('episode')[f'trade_qty_A{i+1}'].sum()
    ax.plot(net_trade.index, net_trade.rolling(ROLL, min_periods=1).mean(),
            color=agent_colors[i], lw=0.9, label=agent_labels[i])
ax.axhline(0, color='black', ls='-', lw=0.5)
ax.set_xlabel('Episode'); ax.set_ylabel('Net Trade (Mt)')
ax.set_title('Net Secondary Position Over Training')
ax.legend(fontsize=6, ncol=2); ax.grid(True, alpha=0.3)

# 2 — Buy vs Sell volume per agent (converged)
ax = axes[0, 1]
buy_vols = []
sell_vols = []
for i in range(n_agents):
    tq = recent_yr[f'trade_qty_A{i+1}']
    buy_vols.append(tq[tq > 0.01].sum() / LAST_N)
    sell_vols.append(abs(tq[tq < -0.01].sum()) / LAST_N)
x = np.arange(n_agents)
ax.bar(x - 0.15, buy_vols, 0.3, color='forestgreen', alpha=0.8, label='Bought')
ax.bar(x + 0.15, sell_vols, 0.3, color='firebrick', alpha=0.8, label='Sold')
ax.set_xticks(x); ax.set_xticklabels([f'A{i+1}' for i in range(n_agents)])
ax.set_ylabel('Mean Volume / Episode (Mt)')
ax.set_title('Buy vs Sell Volume (converged)'); ax.legend(); ax.grid(True, alpha=0.3)

# 3 — Trade position by year within episode (heatmap)
ax = axes[1, 0]
trade_matrix = np.zeros((n_agents, n_years_per_ep))
for i in range(n_agents):
    trade_by_yr = recent_yr.groupby('year')[f'trade_qty_A{i+1}'].mean()
    trade_matrix[i, :len(trade_by_yr)] = trade_by_yr.values
vmax = max(abs(trade_matrix.min()), abs(trade_matrix.max()))
im = ax.imshow(trade_matrix, aspect='auto', cmap='RdYlGn', vmin=-vmax, vmax=vmax)
ax.set_xlabel('Year'); ax.set_ylabel('Agent')
ax.set_yticks(range(n_agents)); ax.set_yticklabels([f'A{i+1}' for i in range(n_agents)])
ax.set_title('Mean Trade Qty by Agent \u00d7 Year (green=buy, red=sell)')
plt.colorbar(im, ax=ax, label='Trade Qty (Mt)')

# 4 — Net secondary cost over training
ax = axes[1, 1]
for i in range(n_agents):
    snet = yr_df.groupby('episode')[f'secondary_net_A{i+1}'].sum()
    ax.plot(snet.index, snet.rolling(ROLL, min_periods=1).mean(),
            color=agent_colors[i], lw=0.9, label=f'A{i+1}')
ax.axhline(0, color='black', ls='-', lw=0.5)
ax.set_xlabel('Episode'); ax.set_ylabel('Net Cost (M\u20ac)')
ax.set_title('Secondary Net Cost Over Training (+paid / -received)')
ax.legend(fontsize=6, ncol=4); ax.grid(True, alpha=0.3)

# 5 — Secondary price premium over auction
ax = axes[2, 0]
premium = recent_yr['secondary_price'] / recent_yr['clearing_price'].replace(0, np.nan)
premium_yr = premium.groupby(recent_yr['year']).agg(['mean', 'std'])
ax.plot(premium_yr.index, premium_yr['mean'], 'b-o', ms=4, lw=1.5)
ax.fill_between(premium_yr.index,
                premium_yr['mean'] - premium_yr['std'],
                premium_yr['mean'] + premium_yr['std'], alpha=0.2)
ax.axhline(1.0, color='red', ls='--', lw=0.8, label='Par (1.0)')
ax.set_xlabel('Year'); ax.set_ylabel('Secondary / Auction Price')
ax.set_title('Secondary Price Premium by Year'); ax.legend(); ax.grid(True, alpha=0.3)

# 6 — Trade qty vs compliance surplus (who trades based on need?)
ax = axes[2, 1]
if 'compliance_surplus_A1' in recent_yr.columns:
    for i in range(n_agents):
        ax.scatter(recent_yr[f'compliance_surplus_A{i+1}'],
                   recent_yr[f'trade_qty_A{i+1}'],
                   alpha=0.04, s=8, color=agent_colors[i], label=f'A{i+1}')
    ax.axhline(0, color='black', ls='-', lw=0.5)
    ax.axvline(0, color='black', ls='-', lw=0.5)
    ax.set_xlabel('Compliance Surplus (Mt)')
    ax.set_ylabel('Trade Qty (Mt, +buy/-sell)')
    ax.set_title('Trading vs Compliance Need')
    ax.legend(fontsize=6, ncol=4)
else:
    ax.text(0.5, 0.5, 'compliance_surplus not logged', ha='center', va='center',
            transform=ax.transAxes)
ax.grid(True, alpha=0.3)

plt.suptitle('A2 \u2014 Secondary Market Strategy', fontsize=14, y=1.01)
plt.tight_layout(); plt.show()

### A3. Green Transition Strategy

Investment patterns in renewable energy:
- **Transition speed** — how quickly does each archetype green its portfolio?
- **Investment intensity** — how aggressively do agents invest?
- **Technology preference** — onshore wind, offshore wind, or solar?
- **Construction pipeline** — queuing effects and timing

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))

# 1 — Green fraction evolution over training
ax = axes[0, 0]
for i in range(n_agents):
    ax.plot(ep_df['episode'], ep_df[f'green_frac_A{i+1}'].rolling(ROLL, min_periods=1).mean(),
            color=agent_colors[i], lw=1, label=agent_labels[i])
ax.set_xlabel('Episode'); ax.set_ylabel('Green Fraction')
ax.set_title('Green Transition Over Training')
ax.legend(fontsize=6, ncol=2); ax.grid(True, alpha=0.3)

# 2 — Green fraction by year within episode (converged)
ax = axes[0, 1]
for i in range(n_agents):
    gf_yr = recent_yr.groupby('year')[f'green_frac_A{i+1}'].mean()
    ax.plot(gf_yr.index, gf_yr.values * 100, color=agent_colors[i], marker='o', ms=3,
            lw=1.2, label=f'A{i+1}')
ax.set_xlabel('Year'); ax.set_ylabel('Green Fraction (%)')
ax.set_title('Within-Episode Green Trajectory (converged)')
ax.legend(fontsize=6, ncol=4); ax.grid(True, alpha=0.3)

# 3 — Delta green by year (when do agents invest?)
ax = axes[0, 2]
for i in range(n_agents):
    dg_yr = recent_yr.groupby('year')[f'delta_green_A{i+1}'].mean() * 100
    ax.plot(dg_yr.index, dg_yr.values, color=agent_colors[i], marker='s', ms=3,
            lw=1, label=f'A{i+1}')
ax.axhline(0, color='black', ls='-', lw=0.5)
ax.set_xlabel('Year'); ax.set_ylabel('\u0394 Green (%)')
ax.set_title('Investment Timing: \u0394Green by Year')
ax.legend(fontsize=6, ncol=4); ax.grid(True, alpha=0.3)

# 4 — Investment cost per agent (converged)
ax = axes[1, 0]
inv_means = [recent_yr.groupby('episode')[f'invest_cost_A{i+1}'].sum().mean()
             for i in range(n_agents)]
ax.bar([f'A{i+1}' for i in range(n_agents)], inv_means, color=agent_colors)
ax.set_ylabel('Mean Investment Cost / Episode (M\u20ac)')
ax.set_title('Total Investment Spend (converged)'); ax.grid(True, alpha=0.3)

# 5 — Construction queue over training
ax = axes[1, 1]
for i in range(n_agents):
    col = f'queue_size_A{i+1}'
    if col in ep_df.columns:
        ax.plot(ep_df['episode'], ep_df[col].rolling(ROLL, min_periods=1).mean(),
                color=agent_colors[i], lw=0.8, label=f'A{i+1}')
ax.set_xlabel('Episode'); ax.set_ylabel('Queue Size')
ax.set_title('Construction Queue Over Training')
ax.legend(fontsize=6, ncol=4); ax.grid(True, alpha=0.3)

# 6 — Green fraction vs emissions (converged)
ax = axes[1, 2]
for i in range(n_agents):
    gf = recent_yr.groupby('episode')[f'green_frac_A{i+1}'].last()
    em = recent_yr.groupby('episode')[f'emissions_A{i+1}'].mean()
    ax.scatter(gf * 100, em, alpha=0.15, s=12, color=agent_colors[i], label=f'A{i+1}')
ax.set_xlabel('Final Green Fraction (%)'); ax.set_ylabel('Mean Emissions (Mt)')
ax.set_title('Green Fraction vs Emissions')
ax.legend(fontsize=6, ncol=4); ax.grid(True, alpha=0.3)

plt.suptitle('A3 \u2014 Green Transition Strategy', fontsize=14, y=1.01)
plt.tight_layout(); plt.show()

### A4. Compliance & Banking Strategy

How agents manage their allowance portfolio:
- **Banking** — accumulating surplus for future compliance
- **Shortfall avoidance** — learning to stay compliant
- **Risk management** — holdings buffer vs cost minimization

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))

# 1 — Holdings trajectory by year (converged)
ax = axes[0, 0]
for i in range(n_agents):
    h_yr = recent_yr.groupby('year')[f'holdings_A{i+1}'].mean()
    ax.plot(h_yr.index, h_yr.values, color=agent_colors[i], marker='o', ms=3,
            lw=1.2, label=f'A{i+1}')
ax.set_xlabel('Year'); ax.set_ylabel('Holdings (Mt)')
ax.set_title('Banking: Year-by-Year Holdings')
ax.legend(fontsize=6, ncol=4); ax.grid(True, alpha=0.3)

# 2 — Shortfall per agent over training
ax = axes[0, 1]
for i in range(n_agents):
    sf = ep_df[f'shortfall_A{i+1}'].rolling(ROLL, min_periods=1).mean()
    ax.plot(ep_df['episode'], sf, color=agent_colors[i], lw=0.8, label=f'A{i+1}')
ax.set_xlabel('Episode'); ax.set_ylabel('Total Shortfall (Mt)')
ax.set_title('Shortfall Reduction Over Training')
ax.legend(fontsize=6, ncol=4); ax.grid(True, alpha=0.3)

# 3 — Holdings vs emissions need (buffer ratio)
ax = axes[0, 2]
for i in range(n_agents):
    h = recent_yr[f'holdings_A{i+1}']
    e = recent_yr[f'emissions_A{i+1}'].replace(0, np.nan)
    buffer = (h / e).dropna()
    buf_yr = buffer.groupby(recent_yr.loc[buffer.index, 'year']).mean()
    ax.plot(buf_yr.index, buf_yr.values, color=agent_colors[i], marker='o', ms=3,
            lw=1, label=f'A{i+1}')
ax.axhline(1.0, color='red', ls='--', lw=0.8, label='Need=1.0')
ax.set_xlabel('Year'); ax.set_ylabel('Holdings / Emissions')
ax.set_title('Bank Buffer Ratio by Year')
ax.legend(fontsize=6, ncol=3); ax.grid(True, alpha=0.3)

# 4 — MAC fuel-switching usage
ax = axes[1, 0]
mac_cols_ep = [f'total_mac_reduction_A{i+1}' for i in range(n_agents)
               if f'total_mac_reduction_A{i+1}' in ep_df.columns]
if mac_cols_ep:
    for i, col in enumerate(mac_cols_ep):
        ax.plot(ep_df['episode'], ep_df[col].rolling(ROLL, min_periods=1).mean(),
                color=agent_colors[i], lw=0.8, label=f'A{i+1}')
    ax.set_xlabel('Episode'); ax.set_ylabel('MAC Reduction (Mt)')
    ax.set_title('Fuel Switching (Coal\u2192Gas)')
    ax.legend(fontsize=6, ncol=4)
else:
    ax.text(0.5, 0.5, 'MAC data not available', ha='center', va='center',
            transform=ax.transAxes)
ax.grid(True, alpha=0.3)

# 5 — Penalty by archetype
ax = axes[1, 1]
pen_by_arch = {}
for i in range(n_agents):
    arch = agent_labels[i].split('(')[1].rstrip(')') if '(' in agent_labels[i] else f'A{i+1}'
    pen = recent_ep[f'penalty_A{i+1}'].mean()
    pen_by_arch.setdefault(arch, []).append(pen)
arch_names = list(pen_by_arch.keys())
arch_means = [np.mean(v) for v in pen_by_arch.values()]
ax.bar(arch_names, arch_means, color=['#555555', '#FF8C00', '#2E8B57', '#1E90FF'][:len(arch_names)])
ax.set_ylabel('Mean Penalty / Episode (M\u20ac)')
ax.set_title('Penalty by Archetype (converged)')
plt.setp(ax.get_xticklabels(), rotation=15, ha='right')
ax.grid(True, alpha=0.3)

# 6 — Emissions trajectory by year (converged)
ax = axes[1, 2]
for i in range(n_agents):
    em_yr = recent_yr.groupby('year')[f'emissions_A{i+1}'].mean()
    ax.plot(em_yr.index, em_yr.values, color=agent_colors[i], marker='o', ms=3,
            lw=1.2, label=f'A{i+1}')
ax.set_xlabel('Year'); ax.set_ylabel('Emissions (Mt)')
ax.set_title('Emission Trajectories Within Episode')
ax.legend(fontsize=6, ncol=4); ax.grid(True, alpha=0.3)

plt.suptitle('A4 \u2014 Compliance & Banking Strategy', fontsize=14, y=1.01)
plt.tight_layout(); plt.show()

### A5. Reward Decomposition & Cost Structure

Decompose the per-agent reward into its main cost components. The
cost-breakdown panel now folds in **collateral cost** alongside the four
original components (auction, secondary, investment, penalty). Older logs
without `collateral_cost_A*` columns simply skip that layer.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))

# 1 — Reward distribution (converged)
ax = axes[0, 0]
rew_data = [recent_ep[f'reward_A{i+1}'].values for i in range(n_agents) if f'reward_A{i+1}' in recent_ep.columns]
if rew_data:
    bp = ax.boxplot(rew_data, tick_labels=[f'A{i+1}' for i in range(len(rew_data))], patch_artist=True, showfliers=False)
    for i, patch in enumerate(bp['boxes']):
        patch.set_facecolor(agent_colors[i])
ax.set_ylabel('Total Reward / Episode')
ax.set_title('Reward Distribution (converged)'); ax.grid(True, alpha=0.3)

# 2 — Cost breakdown per agent (auction / secondary / investment / penalty / collateral)
ax = axes[0, 1]
cost_components = {}
for comp, col_tmpl in [('Auction',     'auction_cost_A{}'),
                       ('SecondaryNet','secondary_net_A{}'),
                       ('Investment',  'invest_cost_A{}'),
                       ('Penalty',     'penalty_A{}'),
                       ('Collateral',  'collateral_cost_A{}')]:
    means = []
    for i in range(n_agents):
        col = col_tmpl.format(i+1)
        if col in recent_yr.columns:
            means.append(recent_yr.groupby('episode')[col].sum().mean())
        else:
            means.append(0)
    cost_components[comp] = means

x = np.arange(n_agents)
bottom = np.zeros(n_agents)
comp_colors = {'Auction': 'steelblue', 'SecondaryNet': 'coral',
               'Investment': 'seagreen', 'Penalty': 'firebrick',
               'Collateral': 'goldenrod'}
for comp, vals in cost_components.items():
    vals_pos = np.maximum(vals, 0)
    ax.bar(x, vals_pos, 0.6, bottom=bottom, label=comp, color=comp_colors[comp], alpha=0.85)
    bottom += vals_pos
ax.set_xticks(x); ax.set_xticklabels([f'A{i+1}' for i in range(n_agents)])
ax.set_ylabel('Cost (M€ / episode)')
ax.set_title('Cost Breakdown by Agent (incl. collateral)'); ax.legend(fontsize=7); ax.grid(True, alpha=0.3)

# 3 — Reward vs green fraction
ax = axes[0, 2]
for i in range(n_agents):
    if f'reward_A{i+1}' not in recent_ep.columns:
        continue
    gf = recent_ep[f'green_frac_A{i+1}']
    rw = recent_ep[f'reward_A{i+1}']
    ax.scatter(gf * 100, rw, alpha=0.15, s=12, color=agent_colors[i], label=f'A{i+1}')
ax.set_xlabel('Final Green Fraction (%)'); ax.set_ylabel('Total Reward')
ax.set_title('Reward vs Green Fraction'); ax.legend(fontsize=6, ncol=4); ax.grid(True, alpha=0.3)

# 4 — Reward by archetype over training
ax = axes[1, 0]
n_per_arch = max(1, n_agents // 4)
for a_idx, arch in enumerate(_base_archetypes):
    agent_ids = list(range(a_idx * n_per_arch, min((a_idx + 1) * n_per_arch, n_agents)))
    if not agent_ids:
        continue
    arch_rew = sum(ep_df[f'reward_A{i+1}'] for i in agent_ids if f'reward_A{i+1}' in ep_df.columns) / len(agent_ids)
    if isinstance(arch_rew, pd.Series):
        ax.plot(ep_df['episode'], arch_rew.rolling(ROLL, min_periods=1).mean(), lw=1.5, label=arch)
ax.set_xlabel('Episode'); ax.set_ylabel('Mean Reward')
ax.set_title('Reward by Archetype'); ax.legend(); ax.grid(True, alpha=0.3)

# 5 — System cost by year (now includes collateral if logged)
ax = axes[1, 1]
_year_cost_specs = [
    ('Auction',    'auction_cost_A{}', 'steelblue'),
    ('Investment', 'invest_cost_A{}',  'seagreen'),
    ('Collateral', 'collateral_cost_A{}', 'goldenrod'),
]
for comp, col_tmpl, color in _year_cost_specs:
    cols = [col_tmpl.format(i+1) for i in range(n_agents) if col_tmpl.format(i+1) in recent_yr.columns]
    if not cols:
        continue
    by_yr = recent_yr.groupby('year')[cols].sum().sum(axis=1)
    ax.plot(by_yr.index, by_yr.values, marker='o', ms=4, lw=1.5, color=color, label=comp)
ax.set_xlabel('Year'); ax.set_ylabel('Total Cost (M€)')
ax.set_title('System Cost by Year'); ax.legend(); ax.grid(True, alpha=0.3)

# 6 — Reward per year within episode
ax = axes[1, 2]
for i in range(n_agents):
    col = f'reward_A{i+1}'
    if col not in recent_yr.columns:
        continue
    rew_yr = recent_yr.groupby('year')[col].mean()
    ax.plot(rew_yr.index, rew_yr.values, color=agent_colors[i], marker='o', ms=3, lw=1, label=f'A{i+1}')
ax.set_xlabel('Year'); ax.set_ylabel('Mean Reward')
ax.set_title('Reward by Year Within Episode'); ax.legend(fontsize=6, ncol=4); ax.grid(True, alpha=0.3)

plt.suptitle('A5 — Reward Decomposition & Cost Structure', fontsize=14, y=1.01)
plt.tight_layout(); plt.show()


### A6. Deterministic Evaluation of Best Agents (optional)

Loads the best checkpoint of each agent for the chosen `(variant, seed)`
and runs one deterministic episode. Skip with `RUN_A6_EVAL = False` if
the sweep was run on a remote machine and only CSV logs are available
locally — A6 is the only deep-dive cell that needs the actual environment.


In [ ]:
RUN_A6_EVAL = False  # set to True to run deterministic evaluation

if not RUN_A6_EVAL:
    print('A6 skipped (RUN_A6_EVAL=False).')
else:
    import numpy as np
    from src.environment.ets_environment import ETSEnvironment
    from scripts.train import build_agents

    # Resolve the checkpoint directory: sweep launcher uses checkpoints_<variant>_s<seed>;
    # fall back to the legacy checkpoints_s<seed>.
    variant_dir = Path(_run['training_log']).parent
    ckpt_candidates = [
        variant_dir / f'checkpoints_{PICK_VARIANT}_s{PICK_SEED}',
        variant_dir / f'checkpoints_s{PICK_SEED}',
    ]
    ckpt_dir = next((p for p in ckpt_candidates if p.exists()), None)
    if ckpt_dir is None:
        print(f'No checkpoint directory found under {variant_dir}.')
    else:
        print(f'Loading best checkpoints from: {ckpt_dir}')
        eval_env = ETSEnvironment(config, seed=0)
        eval_agents = build_agents(eval_env, config, seed=0)
        tech_names = config['technologies']['names']

        for i, agent in enumerate(eval_agents):
            ckpt_path = ckpt_dir / f'agent_{i}_best.pt'
            if ckpt_path.exists():
                agent.load(str(ckpt_path))
                print(f'  loaded agent {i}')
            else:
                print(f'  no checkpoint for agent {i} ({ckpt_path.name})')

        obs1, _ = eval_env.reset(seed=0)
        eval_data = []
        for year in range(config['simulation']['n_years']):
            auction_actions = np.zeros((n_agents, 6), dtype=np.float32)
            for i in range(n_agents):
                action, _, _ = eval_agents[i].select_auction_action(obs1[i], deterministic=True)
                auction_actions[i] = action
            obs2, _ = eval_env.step_auction(auction_actions)

            secondary_actions = np.zeros((n_agents, 2), dtype=np.float32)
            for i in range(n_agents):
                action, _, _ = eval_agents[i].select_secondary_action(obs2[i], deterministic=True)
                secondary_actions[i] = action

            obs1, rewards, terminated, _, info = eval_env.step_secondary(secondary_actions)
            log = info['year_log']

            row = {'year': year, 'cap': log['cap'], 'price': log['clearing_price'],
                   'secondary_price': log.get('secondary_clearing', 0),
                   'tnac': log.get('tnac', 0)}
            for i in range(n_agents):
                row[f'green_A{i+1}'] = log['green_fracs'][i]
                row[f'emissions_A{i+1}'] = log['emissions'][i]
                row[f'alloc_A{i+1}'] = log['allocations'][i]
                row[f'holdings_A{i+1}'] = log['holdings'][i]
                row[f'trade_qty_A{i+1}'] = log['trade_qtys'][i]
                row[f'shortfall_A{i+1}'] = log['shortfalls'][i]
                row[f'reward_A{i+1}'] = log['rewards'][i]
                row[f'invest_cost_A{i+1}'] = log['invest_costs'][i]
                for t in range(5):
                    row[f'mix_A{i+1}_{tech_names[t]}'] = log['tech_mixes'][i][t]
            eval_data.append(row)
            if terminated:
                break

        eval_df = pd.DataFrame(eval_data)
        print(f'Deterministic evaluation: {len(eval_df)} years')
        display(eval_df[['year', 'cap', 'price', 'secondary_price', 'tnac']])

        # Plots
        fig, axes = plt.subplots(2, 3, figsize=(18, 11))
        ax = axes[0, 0]
        ax.plot(eval_df['year'], eval_df['price'], 'b-o', ms=4, label='Auction', lw=1.5)
        ax.plot(eval_df['year'], eval_df['secondary_price'], 'r-s', ms=4, label='Secondary', lw=1.5)
        ax.set_xlabel('Year'); ax.set_ylabel('Price (€/t)')
        ax.set_title('Price Trajectories'); ax.legend(); ax.grid(True, alpha=0.3)

        ax = axes[0, 1]
        for i in range(n_agents):
            ax.plot(eval_df['year'], eval_df[f'trade_qty_A{i+1}'], color=agent_colors[i],
                    marker='o', ms=3, lw=1, label=f'A{i+1}')
        ax.axhline(0, color='black', ls='-', lw=0.5)
        ax.set_xlabel('Year'); ax.set_ylabel('Trade Qty (Mt)')
        ax.set_title('Secondary Trade Patterns'); ax.legend(fontsize=6, ncol=4); ax.grid(True, alpha=0.3)

        ax = axes[0, 2]
        for i in range(n_agents):
            ax.plot(eval_df['year'], eval_df[f'holdings_A{i+1}'], color=agent_colors[i],
                    marker='o', ms=3, lw=1, label=f'A{i+1}')
        ax.set_xlabel('Year'); ax.set_ylabel('Holdings (Mt)')
        ax.set_title('Banking: Allowance Holdings'); ax.legend(fontsize=6, ncol=4); ax.grid(True, alpha=0.3)

        ax = axes[1, 0]
        for i in range(n_agents):
            ax.plot(eval_df['year'], eval_df[f'emissions_A{i+1}'], color=agent_colors[i], lw=1.2)
            ax.plot(eval_df['year'], eval_df[f'alloc_A{i+1}'], color=agent_colors[i], ls='--', lw=0.8, alpha=0.6)
        ax.set_xlabel('Year'); ax.set_ylabel('Mt')
        ax.set_title('Emissions (solid) vs Allocation (dashed)'); ax.grid(True, alpha=0.3)

        ax = axes[1, 1]
        for i in range(n_agents):
            ax.plot(eval_df['year'], eval_df[f'green_A{i+1}'] * 100, color=agent_colors[i],
                    marker='o', ms=3, lw=1.2, label=f'A{i+1}')
        ax.set_xlabel('Year'); ax.set_ylabel('Green Fraction (%)')
        ax.set_title('Green Transition'); ax.legend(fontsize=6, ncol=4); ax.grid(True, alpha=0.3)

        ax = axes[1, 2]
        for i in range(n_agents):
            cum_rew = eval_df[f'reward_A{i+1}'].cumsum()
            ax.plot(eval_df['year'], cum_rew, color=agent_colors[i], lw=1.5, label=f'A{i+1}')
        ax.set_xlabel('Year'); ax.set_ylabel('Cumulative Reward')
        ax.set_title('Cumulative Reward'); ax.legend(fontsize=6, ncol=4); ax.grid(True, alpha=0.3)

        plt.suptitle('A6 — Deterministic Evaluation', fontsize=14, y=1.01)
        plt.tight_layout(); plt.show()
